# E4: Pipeline Control and Component Ablation

**Work Plan Reference:** Deliverable III · F26-066 (Mixture of Experts Framework for Domain Adaptive Colorization)
**Owner:** Person B (Extraction, Colorization Wrapper, Blending, Config File, and E4 Experiments)

---

### Experiment E4 Overview
Evaluate the full pipeline with identical experts compared with the E3 baseline, changing one component at a time:
1. **Identical Experts:** Four domain slots (`people`, `nature`, `urban`, `animals`) plus `fallback`, all loading the same pretrained DDColor-T model (ConvNeXt-T backbone, 256×256 input, ab chrominance output).
2. **Context Margin Ablation:** Expanding bounding boxes by context margin $\rho \in \{0.0, 0.1, 0.2, 0.3\}$.
3. **Blending Switch:** Soft Gaussian-weighted blending (`blending_on=True`) versus hard binary mask paste (`blending_on=False`).
4. **Smoothing Width Ablation:** Gaussian standard deviation $\sigma_0 \in \{2.0, 5.0, 10.0, 15.0, 20.0\}$ px.
5. **Region Shape:** SAM masks (`region_shape='mask'`) versus bounding boxes (`region_shape='box'`).
6. **E3 Baseline Comparison:** DDColor-T on whole image without decomposition.

**Metrics Evaluated:** PSNR, SSIM, Lab $\Delta E$, Boundary $\Delta E$, Colorfulness, LPIPS.

---

### Kaggle Settings (Right-Hand Panel):
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** **ON** (required for pip install & HuggingFace weights download)
- **Dataset:** In Kaggle click **Add Data -> Datasets** and search for [`dntai2/cocostuf-2017`](https://www.kaggle.com/datasets/dntai2/cocostuf-2017) (mounted at `/kaggle/input/cocostuf-2017`).

## 1. Clone or Update Repository

In [ ]:
import os
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/zaeem886/domain-adaptive-colorization.git"
BRANCH = "experiment_code"  # E4 runner and notebook live on this branch
REPO_DIR = Path("/kaggle/working/domain-adaptive-colorization")

if (REPO_DIR / ".git").is_dir():
    print("Updating existing repository...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} (branch {BRANCH})...")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())


## 2. Install Dependencies

Kaggle environments provide PyTorch and Transformers. Here we install `lpips`, `segment-anything`, `basicsr`, `scikit-image`, and `huggingface-hub`.

In [ ]:
!pip install -q --no-deps lpips
!pip install -q segment-anything transformers timm einops scikit-image scipy opencv-python-headless huggingface-hub matplotlib pandas
print("Dependencies installed successfully.")

## 3. Dataset Setup (Symlink COCO val2017)

The dataset [`dntai2/cocostuf-2017`](https://www.kaggle.com/datasets/dntai2/cocostuf-2017) is mounted at `/kaggle/input/cocostuf-2017`.
We create fast symlinks so that no data is copied into working storage.

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/cocostuf-2017")
DATASET_ROOT = Path("/kaggle/working/cocostuff_data")
IMAGES_ROOT = DATASET_ROOT / "images"

if SRC.is_dir():
    val_target = SRC / "val2017" / "val2017"
    if not val_target.is_dir():
        val_target = SRC / "val2017"
    link = IMAGES_ROOT / "val2017"
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    link.symlink_to(val_target, target_is_directory=True)
    print("Linked val2017 images to:", link)
else:
    print("Notice: /kaggle/input/cocostuf-2017 not mounted. If running locally or on custom dataset, set IMAGES_ROOT directly.")
    IMAGES_ROOT = Path("pics")

## 4. Download DDColor and Pretrained DDColor-T Model

We clone the official DDColor codebase and fetch the official `piddnad/ddcolor_paper_tiny` pretrained weights (`pytorch_model.bin`) from Hugging Face Hub.

In [ ]:
from huggingface_hub import hf_hub_download
import urllib.request

DDCOLOR_DIR = Path("/kaggle/working/DDColor")
if not DDCOLOR_DIR.exists():
    !git clone --depth 1 https://github.com/piddnad/DDColor.git {DDCOLOR_DIR}

print("Downloading DDColor-T checkpoint from Hugging Face...")
CHECKPOINT = hf_hub_download("piddnad/ddcolor_paper_tiny", "pytorch_model.bin")
print("DDColor repository:", DDCOLOR_DIR)
print("DDColor-T checkpoint:", CHECKPOINT)

# Optional: download SAM ViT-B checkpoint if online Grounded-SAM decomposition is needed
SAM_CHECKPOINT = Path("/kaggle/working/sam_vit_b.pth")
if not SAM_CHECKPOINT.is_file():
    print("Downloading SAM ViT-B weights (~375MB)... ")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        str(SAM_CHECKPOINT),
    )
    print("SAM weights downloaded:", SAM_CHECKPOINT)
else:
    print("SAM checkpoint exists:", SAM_CHECKPOINT)

## 5. Common Configuration and Command Runner

Shared CLI runner wrapping `experiments/e4.py`.

In [ ]:
import shlex
import sys

# Flags understood by the `run` subcommand. e4.py only defines these on
# `run`/`all`, so they must not be passed to `make-lists` or `figures`.
RUN_COMMON = [
    "--checkpoint", str(CHECKPOINT),
    "--ddcolor-path", str(DDCOLOR_DIR),
    "--images-root", str(IMAGES_ROOT),
    "--val-split", "val2017",
    "--cache-dir", "cache",
    "--sam-checkpoint", str(SAM_CHECKPOINT),
    "--use-sam",
    "--device", "cuda",
]

def run_e4(stage, *stage_args):
    """Run one experiments/e4.py stage with stage-appropriate arguments.

    ``--results-dir`` must precede ``run`` (root parser option) and follow
    ``figures`` (the figures subparser defines its own). ``make-lists`` only
    needs ``--data-lists-dir``.
    """
    if stage == "run":
        command = [
            sys.executable, "experiments/e4.py",
            "--results-dir", "results/e4",
            "run", *stage_args, *RUN_COMMON,
        ]
    elif stage == "figures":
        command = [
            sys.executable, "experiments/e4.py",
            "figures", "--results-dir", "results/e4",
        ]
    elif stage == "make-lists":
        command = [
            sys.executable, "experiments/e4.py",
            "--data-lists-dir", "data_lists",
            "make-lists", *stage_args,
        ]
    else:
        raise ValueError(f"Unsupported stage: {stage}")

    print("$", " ".join(shlex.quote(str(part)) for part in command))
    subprocess.run(command, check=True)


## 6. Stage 1: Create Fixed Test ID Lists

Exports the fixed 100-image evaluation set (20 per domain: people, nature, urban, animals, plus 20 mixed scenes) to `data_lists/e4_test_ids.txt`.

In [ ]:
run_e4("make-lists")

## 7. Stage 2: Quick Smoke Test (`--limit 2`)

Validates the entire pipeline end-to-end on GPU with 2 test images before launching the full run.

In [ ]:
run_e4("run", "--limit", "2", "--save-images")

## 8. Stage 3: Full E4 Experiment Execution

Runs all E4 conditions (default pipeline, context margins 0.0, 0.1, 0.2, 0.3, blending off, smoothing widths, bounding boxes, and E3 whole-image baseline).

*Note: Results are appended row-by-row and are fully resumable; re-running resumes from where it left off.*

In [ ]:
# Run the full experiment across the fixed 100-image E4 list.
# `--save-images` must be present on the run that computes the metrics;
# a later re-run skips completed pairs and saves nothing.
run_e4(
    "run",
    "--save-images",
    "--id-list", "data_lists/e4_test_ids.txt",
)


## 9. Record Run Environment and Manifest

Writes the versions, GPU, git commit and checkpoint sizes to `results/e4/environment.txt` so they travel with the results bundle and can be quoted in the report's experimental-setup section.


In [ ]:
import importlib.metadata
import platform
import subprocess

PACKAGES = (
    "torch", "torchvision", "transformers", "segment-anything", "lpips",
    "numpy", "pillow", "opencv-python-headless", "scikit-image", "scipy",
    "matplotlib", "pandas",
)


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


lines = [
    f"python: {sys.version.split()[0]}",
    f"platform: {platform.platform()}",
]
for name in PACKAGES:
    lines.append(f"{name}: {package_version(name)}")

try:
    import torch

    lines.append(f"torch.cuda.available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        lines.append(f"gpu: {torch.cuda.get_device_name(0)}")
        lines.append(f"gpu_count: {torch.cuda.device_count()}")
        lines.append(f"cuda: {torch.version.cuda}")
except Exception as exc:  # keep the manifest usable even if torch fails
    lines.append(f"torch.cuda: unavailable ({exc})")

branch = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True,
).stdout.strip()
lines.extend([f"branch: {branch}", f"commit: {commit}"])

for label, path in (
    ("ddcolor_checkpoint", CHECKPOINT),
    ("sam_checkpoint", SAM_CHECKPOINT),
):
    p = Path(path)
    if p.is_file():
        lines.append(f"{label}: {p} ({p.stat().st_size / 1e6:.1f} MB)")
    else:
        lines.append(f"{label}: {p} (not found)")

report = "\n".join(lines)
print(report)
Path("results/e4").mkdir(parents=True, exist_ok=True)
Path("results/e4/environment.txt").write_text(report + "\n", encoding="utf-8")


## 10. Validate Run Completeness

Fails loudly if any condition is missing an image, any (condition, image) pair has fewer than six metric rows, or the decomposition cache is incomplete. An incomplete cache is the signature of GroundedSAM silently falling back to a whole-image region, which would invalidate the E4 comparison.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(".").resolve()))
from colorize.regions import load_decomposition

RESULTS = Path("results/e4")
ID_LIST = Path("data_lists/e4_test_ids.txt")

EXPECTED_CONDITIONS = {
    "default",
    "blending_off",
    "shape_box",
    "e3_ddcolor_t_whole",
    "margin_0.0", "margin_0.1", "margin_0.2", "margin_0.3",
    "smooth_2.0", "smooth_5.0", "smooth_10.0", "smooth_15.0", "smooth_20.0",
}
EXPECTED_METRICS = {
    "psnr", "ssim", "lab_delta_e", "boundary_delta_e", "colorfulness", "lpips",
}

df = pd.read_csv(RESULTS / "results.csv", dtype={"image_id": str})
expected_ids = {
    f"{int(line):012d}"
    for line in ID_LIST.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.startswith("#")
}

per_cond = df.groupby("condition")["image_id"].nunique()
per_pair = df.groupby(["condition", "image_id"]).size()
metrics = sorted(df["metric"].unique())
conds = sorted(df["condition"].unique())
duplicates = int(df.duplicated(["condition", "image_id", "metric"]).sum())

decomp_paths = {}
for cache_subdir in ("e4_decomp", "e1_test"):
    for path in sorted((Path("cache") / cache_subdir).glob("*.npz")):
        decomp_paths.setdefault(path.stem, path)
decomps = list(decomp_paths.values())
empty_regions = sum(
    1 for path in decomps
    if len(load_decomposition(str(path)).regions) == 0
)

report = "\n".join([
    f"rows: {len(df)}",
    f"conditions ({len(conds)}): {', '.join(conds)}",
    f"metrics ({len(metrics)}): {', '.join(metrics)}",
    f"expected images: {len(expected_ids)}",
    f"images per condition: {per_cond.to_dict()}",
    f"metrics per (condition, image): {sorted(per_pair.unique().tolist())}",
    f"duplicate rows: {duplicates}",
    f"cached decompositions: {len(decomps)} ({empty_regions} with zero regions)",
])
print(report)
(RESULTS / "validation.txt").write_text(report + "\n", encoding="utf-8")

assert set(conds) == EXPECTED_CONDITIONS, (
    f"condition set mismatch; missing={EXPECTED_CONDITIONS - set(conds)}, "
    f"unexpected={set(conds) - EXPECTED_CONDITIONS}"
)
assert set(metrics) == EXPECTED_METRICS, (
    f"metric set mismatch; missing={EXPECTED_METRICS - set(metrics)}, "
    f"unexpected={set(metrics) - EXPECTED_METRICS}"
)
assert (per_cond == len(expected_ids)).all(), (
    f"some conditions do not cover all {len(expected_ids)} images: {per_cond.to_dict()}"
)
assert set(df["image_id"]) == expected_ids, "CSV image IDs do not match e4_test_ids.txt"
assert (per_pair == 6).all(), "some (condition, image) pairs have partial metric rows"
assert duplicates == 0, f"{duplicates} duplicate (condition, image, metric) rows"
assert len(decomps) == len(expected_ids), (
    "decomposition cache incomplete: GroundedSAM may have fallen back to whole-image"
)


## 11. Stage 4: Results Summary Table

Display quantitative metric averages across all experimental conditions.

In [ ]:
import pandas as pd

csv_path = Path("results/e4/results.csv")
if csv_path.is_file():
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} total metric records.")
    
    # Pivot mean by condition and metric
    summary_df = df.groupby(["condition", "metric"])["value"].mean().unstack()
    display(summary_df.round(4))
else:
    print("Results CSV not found yet.")

## 12. Stage 5: Generate Figures and Markdown Analysis Report

In [ ]:
run_e4("figures")

## 13. Stage 6: Display Generated Charts and Qualitative Visual Grids

In [ ]:
from IPython.display import Image as IPImage, Markdown, display

fig_dir = Path("results/e4")

for plot_name in (
    "ablation_margins.png",
    "ablation_smoothing.png",
    "ablation_components.png",
    "comparison_e3_e4.png",
    "e4_examples_grid.png",
):
    p = fig_dir / plot_name
    if p.is_file():
        print(f"=== {plot_name} ===")
        display(IPImage(filename=str(p)))

notes_file = fig_dir / "e4_notes.md"
if notes_file.is_file():
    display(Markdown(notes_file.read_text(encoding="utf-8")))

## 14. Stage 7: Download the Complete Experiment Results Bundle

Packages `results/e4` (CSV records, summary tables, plots, report, environment and validation logs) and `data_lists` into `/kaggle/working/e4_results.zip`. To keep the archive manageable, only up to five saved images for the six report conditions are included; the full image set stays in the Kaggle working directory.


In [ ]:
staging = Path("/kaggle/working/e4_bundle")
shutil.rmtree(staging, ignore_errors=True)
staging.mkdir(parents=True, exist_ok=True)

# Copy results without the (large) rendered images, then add a small,
# report-relevant subset: the conditions used in the qualitative grid.
if Path("results/e4").is_dir():
    shutil.copytree(
        "results/e4",
        staging / "results/e4",
        ignore=shutil.ignore_patterns("images"),
    )

KEEP_CONDITIONS = {
    "default",
    "blending_off",
    "shape_box",
    "e3_ddcolor_t_whole",
    "margin_0.0",
    "margin_0.3",
}
KEEP_PER_CONDITION = 5

images_src = Path("results/e4/images")
images_dst = staging / "results/e4/images"
if images_src.is_dir():
    for cond_dir in sorted(images_src.iterdir()):
        if not cond_dir.is_dir() or cond_dir.name not in KEEP_CONDITIONS:
            continue
        out_dir = images_dst / cond_dir.name
        out_dir.mkdir(parents=True, exist_ok=True)
        for png in sorted(cond_dir.glob("*.png"))[:KEEP_PER_CONDITION]:
            shutil.copy2(png, out_dir / png.name)

if Path("data_lists").is_dir():
    shutil.copytree("data_lists", staging / "data_lists")

archive = shutil.make_archive("/kaggle/working/e4_results", "zip", root_dir=staging)
print(f"Results archive ready: {archive} ({Path(archive).stat().st_size / 1e6:.2f} MB)")
